In [ ]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    )
from core_helpers import (
    init_world_state
)

target_symbol = "TROOTS-1"

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
state = await init_world_state(fleet_api, agents_api, systems_api)
traits = state.traits.by_wp
print(f"[BOOT] fleet={len(state.fleet.by_symbol)} ships, waypoints={len(state.waypoints.by_symbol)} (system), traits={sum(len(v) for v in state.traits.by_wp.values())}")
async def get_closest_wp_by_trait(trait: str):

    state = await init_world_state(fleet_api, agents_api, systems_api)
    traits = state.traits.by_wp
    data = []
    for wp_symbol, rows_list in traits.items():
        for row in rows_list:
            if row.trait_symbol == "MARKETPLACE":
                x = row.x or 0
                y = row.y or 0
                data.append({"waypoint": wp_symbol, "x": x, "y": y, "distance": math.hypot(x, y)})
    data = pd.DataFrame(data).sort_values("distance", ascending=True).head(25).reset_index(drop=True)
    #print (data.to_string(index=False))
    return data

markets_df = await get_closest_wp_by_trait("MARKETPLACE")
#print(markets_df)
print(markets_df.to_string(index=False))

"""
nav_resp = await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H53")
print("This is nav_resp: ", nav_resp)
print("Sleeping for 4 seconds")
await asyncio.sleep(4)
nav_resp2 = await api_get_ship_nav(fleet_api, "TROOTS-1")
print("This is the current ship status: ", nav_resp2.status)
"""
#await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H52")
"""
market_data = await capture_market_for_waypoint(systems_api,"X1-Q51-H53")
print(market_data)
"""


[BOOT] fleet=2 ships, waypoints=87 (system), traits=240
   waypoint   x   y  distance
  X1-Q51-A1   3  23 23.194827
  X1-Q51-A2   3  23 23.194827
  X1-Q51-A4   3  23 23.194827
  X1-Q51-A3   3  23 23.194827
X1-Q51-AB5A -13  26 29.068884
 X1-Q51-H52 -45   1 45.011110
 X1-Q51-H53 -45   1 45.011110
 X1-Q51-H51 -45   1 45.011110
 X1-Q51-H54 -45   1 45.011110
 X1-Q51-E46  50 -24 55.461698
 X1-Q51-E47  50 -24 55.461698
 X1-Q51-G50 -39 -53 65.802736
 X1-Q51-F48  65  42 77.388630
 X1-Q51-F49  65  42 77.388630
 X1-Q51-D42 -56  62 83.546394


'\nmarket_data = await capture_market_for_waypoint(systems_api,"X1-Q51-H53")\nprint(market_data)\n'

In [4]:

from market_runtime import capture_market_for_waypoint
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3

async def market_to_db(waypoint: str) -> None:
    # Fetch first (network I/O), then write to DB (short-lived connection)
    rows = await capture_market_for_waypoint(systems_api, waypoint)
    with sqlite3.connect("spacetraders.db") as conn:
        snapshot_many(conn, "market_goods", MarketGoodRow, rows["goods"])  # append-only history
        if rows["transactions"]:
            upsert_many(conn, TableSpec(table="market_transactions", pk="id"), rows["transactions"])


# --- main patrol loop ---
async def patrol_markets(ship_symbol: str, markets_df) -> None:
    # dedupe and coerce to plain list of strings
    waypoints: List[str] = list(dict.fromkeys(markets_df["waypoint"].astype(str).tolist()))
    print(waypoints)
    if not waypoints:
        print("[WARN] No waypoints to patrol.")
        return

    print(f"[PATROL] {ship_symbol} looping through {len(waypoints)} markets.")
    idx = 0
    while True:
        wp = waypoints[idx]
        try:
            print(f"[PATROL] -> Navigating to {wp} (#{idx+1}/{len(waypoints)})")
            nav_resp = await api_navigate_ship(fleet_api, ship_symbol, wp)
            print(f"[MARKET] Capturing {wp} …")
            await market_to_db(wp)
            await asyncio.sleep(1.0)  # small dwell

        except asyncio.CancelledError:
            raise
        except Exception as e:
            print(f"[ERR] Patrol step at {wp} failed: {e!r}")
            await asyncio.sleep(3.0)  # brief backoff

        # round-robin
        idx = (idx + 1) % len(waypoints)

# --- kick it off ---
await patrol_markets("TROOTS-1", markets_df)

['X1-Q51-A1', 'X1-Q51-A2', 'X1-Q51-A4', 'X1-Q51-A3', 'X1-Q51-AB5A', 'X1-Q51-H52', 'X1-Q51-H53', 'X1-Q51-H51', 'X1-Q51-H54', 'X1-Q51-E46', 'X1-Q51-E47', 'X1-Q51-G50', 'X1-Q51-F48', 'X1-Q51-F49', 'X1-Q51-D42']
[PATROL] TROOTS-1 looping through 15 markets.
[PATROL] -> Navigating to X1-Q51-A1 (#1/15)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Prep complete
TROOTS-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 15.765788
Arrived and ready
[MARKET] Capturing X1-Q51-A1 …
[PATROL] -> Navigating to X1-Q51-A2 (#2/15)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
TROOTS-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 15.78129
Arrived and ready
[MARKET] Capturing X1-Q51-A2 …
[PATROL] -> Navigating to X1-Q51-A4 (#3/15)
starting na

CancelledError: 